# Week 3 Deep Analysis — Baseline Setup

**Owner:** Member 3 — Jaykisan Saharan  
**Day:** Monday — Baseline and setup

Purpose: initialize the analysis environment, resolve repository-relative paths, and load/inspect the existing order-level KPI exports. These saved exports are interim artifacts; do not treat their values as final until they are reconciled with the validated SQL outputs.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Resolve the repository root whether Jupyter starts in the repo root or notebooks/.
CWD = Path.cwd().resolve()
if (CWD / 'data' / 'interim').exists():
    REPO_ROOT = CWD
elif (CWD.parent / 'data' / 'interim').exists():
    REPO_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        'Could not locate data/interim. Start Jupyter from the repository root or notebooks/.'
    )

INTERIM_DIR = REPO_ROOT / 'data' / 'interim'
print(f'Repository root: {REPO_ROOT}')
print(f'Interim data directory: {INTERIM_DIR}')

In [ ]:
# Load the current saved order-level KPI exports using relative paths.
orders_path = INTERIM_DIR / 'kpi_orders.csv'
orders_all_path = INTERIM_DIR / 'kpi_orders_all.csv'

for path in (orders_path, orders_all_path):
    if not path.exists():
        raise FileNotFoundError(f'Required export not found: {path}')

orders = pd.read_csv(orders_path, low_memory=False)
orders_all = pd.read_csv(orders_all_path, low_memory=False)

print('kpi_orders.csv:', orders.shape)
print('kpi_orders_all.csv:', orders_all.shape)

## Initial structural inspection
Check the columns, data types, and sample rows before calculating new metrics. Keep this first pass descriptive and avoid silently changing source exports.

In [ ]:
display(orders.head())
display(orders.dtypes.rename('dtype').to_frame())
display(orders_all.head())
display(orders_all.dtypes.rename('dtype').to_frame())

In [ ]:
# Basic integrity checks for the order-level exports.
def inspect_export(df: pd.DataFrame, name: str) -> pd.DataFrame:
    summary = pd.DataFrame({
        'check': ['rows', 'columns', 'duplicate_order_ids', 'missing_order_id'],
        'value': [
            len(df),
            len(df.columns),
            int(df['order_id'].duplicated().sum()) if 'order_id' in df.columns else None,
            int(df['order_id'].isna().sum()) if 'order_id' in df.columns else None,
        ],
    })
    print(f'\n{name}')
    display(summary)
    return df.isna().sum().sort_values(ascending=False).rename('missing_values').to_frame()

missing_orders = inspect_export(orders, 'kpi_orders.csv')
missing_orders_all = inspect_export(orders_all, 'kpi_orders_all.csv')
display(missing_orders.head(15).rename_axis('column'))
display(missing_orders_all.head(15).rename_axis('column'))

In [ ]:
# Inspect order statuses and date coverage without assuming every order was delivered.
if 'order_status' in orders_all.columns:
    display(orders_all['order_status'].value_counts(dropna=False).rename_axis('order_status').to_frame('orders'))

if 'order_purchase_timestamp' in orders_all.columns:
    purchase_ts = pd.to_datetime(orders_all['order_purchase_timestamp'], errors='coerce')
    print('Unparseable purchase timestamps:', int(purchase_ts.isna().sum()))
    print('Earliest valid purchase timestamp:', purchase_ts.min())
    print('Latest valid purchase timestamp:', purchase_ts.max())

In [ ]:
# Compare exports at a high level. Differences are evidence to investigate, not an automatic error.
comparison = pd.DataFrame({
    'export': ['kpi_orders.csv', 'kpi_orders_all.csv'],
    'rows': [len(orders), len(orders_all)],
    'unique_order_ids': [orders['order_id'].nunique() if 'order_id' in orders else None, orders_all['order_id'].nunique() if 'order_id' in orders_all else None],
})
display(comparison)

if 'order_id' in orders.columns and 'order_id' in orders_all.columns:
    ids_orders = set(orders['order_id'].dropna().astype(str))
    ids_all = set(orders_all['order_id'].dropna().astype(str))
    print('IDs only in kpi_orders.csv:', len(ids_orders - ids_all))
    print('IDs only in kpi_orders_all.csv:', len(ids_all - ids_orders))

print('Inspection complete. Do not publish final KPIs from these exports until they are reconciled with validated SQL outputs.')